# Analyse exploratoire — UltraMedical Preference

### Synthèse & observations

**Structure**
- Dataset : UltraMedical Preference
- Langue principale : anglais
- Format : dataset de préférences pour DPO
- Taille : 109 353 exemples (train)
- Colonnes principales : `prompt`, `chosen`, `rejected`, `label_type`

**Qualité des données**
- Aucune valeur manquante détectée
- De nombreux prompts apparaissent plusieurs fois, ce qui est cohérent avec un dataset DPO (comparaisons multiples pour un même prompt)
- Aucun nettoyage automatique des doublons de prompts n’est appliqué

**Contenu**
- Dataset conçu pour l’alignement par préférences (DPO), et non pour du SFT classique
- Principaux types de préférences :
  - `hard`
  - `length`
  - `easy`

**Longueur des textes**
- Prompts relativement longs (moyenne ~491 caractères)
- Présence de prompts très détaillés (jusqu’à ~5500 caractères)

**Décisions de préparation**
- Conserver uniquement les champs utiles au DPO (`prompt`, `chosen`, `rejected`)
- Extraire uniquement le contenu textuel des réponses
- Sous-échantillonner le dataset pour un entraînement plus léger et adapté au POC

## 1. Chargement du dataset

In [13]:
import pandas as pd
from datasets import load_dataset

ultra_dataset = load_dataset("TsinghuaC3I/UltraMedical-Preference")
ultra_dataset

DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 109353
    })
    validation: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 2232
    })
    test: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 777
    })
})

## 2. Exploration de la structure

In [3]:
df = pd.DataFrame(ultra_dataset["train"])

print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

df.head(5)

Shape : (109353, 7)
Colonnes : ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback']


,prompt_id,label_type,prompt,chosen,rejected,metadata,feedback
0,"WikiInstruct,8304",length,Investigate the intricacies of immunometabolis...,[{'content': 'Investigate the intricacies of i...,[{'content': 'Investigate the intricacies of i...,"{'golden_answer': None, 'chosen': {'model': 'M...",Assistant A provides a detailed and comprehens...
1,"WikiInstruct,8846",length,Examine the differential immunotoxic effects e...,[{'content': 'Examine the differential immunot...,[{'content': 'Examine the differential immunot...,"{'golden_answer': None, 'chosen': {'model': 'M...","Assistant A's response is well-structured, pro..."
2,"MedMCQA,11404",length,A lady comes with melanotic pigmentation of li...,[{'content': 'A lady comes with melanotic pigm...,[{'content': 'A lady comes with melanotic pigm...,"{'golden_answer': 'A', 'chosen': {'model': 'Me...",Both Assistant A and Assistant B provided answ...
3,"ChatDoctor,33567",easy,my daughter was diagnosed with a liver cyst th...,[{'content': 'my daughter was diagnosed with a...,[{'content': 'my daughter was diagnosed with a...,"{'golden_answer': 'Hi, Thank you for your quer...",Assistant A provides a comprehensive and detai...
4,"WikiInstruct,14624",length,Investigate the physiological mechanisms under...,[{'content': 'Investigate the physiological me...,[{'content': 'Investigate the physiological me...,"{'golden_answer': None, 'chosen': {'model': 'M...",Both Assistant A and Assistant B provide detai...


## 3. Vérification des valeurs manquantes

In [7]:
df.isnull().sum()

prompt_id     0
label_type    0
prompt        0
chosen        0
rejected      0
metadata      0
feedback      0
dtype: int64

## 4. Vérification des doublons

In [8]:
print("Doublons sur prompt :", df.duplicated(subset=["prompt"]).sum())
print("Doublons sur prompt_id :", df.duplicated(subset=["prompt_id"]).sum())

Doublons sur prompt : 32829
Doublons sur prompt_id : 32307


In [9]:
duplicate_prompts = df[df.duplicated(subset=["prompt"], keep=False)]
duplicate_prompts[["prompt", "label_type"]].head(10)

,prompt,label_type
2,A lady comes with melanotic pigmentation of li...,length
3,my daughter was diagnosed with a liver cyst th...,easy
4,Investigate the physiological mechanisms under...,length
5,A 22-year-old medical student presents to a co...,hard
7,In a 50-year-old woman with a history of oral ...,length
9,A 52 year old female complains of sudden visua...,length
12,Context: To demonstrate that interference micr...,hard
16,Acalculous cholecystitis can be seen in all th...,hard
18,"In the context of a normal vaginal delivery, w...",length
23,All the following are true about Papilledema e...,length


## 5. Analyse des types de préférences

In [10]:
df["label_type"].value_counts()

label_type
hard      45765
length    41130
easy      22458
Name: count, dtype: int64

## 6. Analyse de la longueur des prompts

In [11]:
df["prompt_length"] = df["prompt"].str.len()
df["prompt_length"].describe()

count    109353.000000
mean        490.566852
std         341.991481
min          24.000000
25%         286.000000
50%         402.000000
75%         627.000000
max        5509.000000
Name: prompt_length, dtype: float64